In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

1. Loading Datasets

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

In [ ]:
def preprocess_features(df):
    """
    Applied feature engineering to biological and lifestyle metrics.
    Targets cognitive performance prediction.
    """
    df = df.fillna(0)
    
    # Feature 1: Mental Workload (Stress * Working Hours)
    df['mental_workload'] = df['stres_skoru'] * df['gunluk_calisma_saati']
    
    # Feature 2: Sleep Restoration Power (REM + Deep Sleep weighted by Sleep Latency)
    df['sleep_restoration_power'] = (df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']) - (df['uykuya_dalma_suresi_dk'] / 5)
    
    # Feature 3: Sleep Sabotage Index (Caffeine * Screen Time)
    df['sleep_sabotage_index'] = df['uyku_oncesi_kafein_mg'] * df['uyku_oncesi_ekran_suresi_dk']
    
    # Feature 4: Nap Efficiency (Binary flag for 15-45 mins naps)
    df['is_efficient_nap'] = df['sekerleme_suresi_dk'].apply(lambda x: 1 if 15 <= x <= 45 else 0)
    
    # Feature 5: Physical Efficiency (Steps normalized by Resting Heart Rate)
    df['physical_efficiency'] = df['gunluk_adim_sayisi'] / (df['dinlenik_nabiz_bpm'] + 1)
    
    # Feature 6: High Risk Occupational Groups
    risk_groups = ['Saglik Personeli', 'Ogrenci', 'Driver']
    df['occupational_risk_flag'] = df['meslek'].apply(lambda x: 1 if x in risk_groups else 0)

    # Categorical Data Handling
    cat_cols = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)
        
    return df, cat_cols

Apply preprocessing

In [ ]:
train, cat_features = preprocess_features(train)
test, _ = preprocess_features(test)

Preparing Train/Test splits

In [ ]:
X = train.drop(['id', 'bilissel_performans_skoru'], axis=1)
y = train['bilissel_performans_skoru']
X_test = test.drop(['id'], axis=1)

2. CatBoost Hyperparameters (Optimized for GPU)

In [ ]:
cb_params = {
    'iterations': 12000,
    'learning_rate': 0.005,
    'depth': 8,
    'l2_leaf_reg': 20,
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'early_stopping_rounds': 600,
    'task_type': 'GPU',
    'devices': '0',
    'verbose': 1000
}

3. Cross-Validation (K-Fold) Training

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

In [ ]:
print("Starting 5-Fold Cross-Validation...")

In [ ]:
for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**cb_params)
    model.fit(
        X_tr, y_tr, 
        eval_set=(X_val, y_val), 
        use_best_model=True, 
        cat_features=cat_features
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / kf.n_splits
    
    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"Fold {fold+1} Validation RMSE: {fold_rmse:.6f}")

Final Metrics

In [ ]:
total_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\nFinal Out-of-Fold (OOF) RMSE: {total_rmse:.6f}")

4. Generating Submission File

In [ ]:
final_preds = np.clip(test_preds, y.min(), y.max())
output_file = 'submission_catboost_v16.csv'
pd.DataFrame({
    'id': test['id'], 
    'bilissel_performans_skoru': final_preds
}).to_csv(output_file, index=False)

In [ ]:
print(f"Process complete. Submission file saved as: {output_file}")